# RAG Scale Lab on Colab

Runs the retrieval stages on a GPU runtime with high RAM (A100 or L4; the one-million Ukrainian cells need about 17 GB). No secret is needed here: the generation stage uses an OpenRouter key and runs locally from the downloaded `results/`.

Work is kept in `/content`, which is erased when the runtime is recycled; in that case run the notebook again from the top. Within one session finished stages and cells are skipped. Each stage ends by downloading the small `results/` folder.

In [ ]:
REPOSITORY = "https://github.com/dbulgakov/ua-rag-scale"
OUT = "/content/out"

%env HF_HUB_DISABLE_PROGRESS_BARS=1
%env HF_DATASETS_DISABLE_PROGRESS_BARS=1
!git clone -q $REPOSITORY /content/ua-rag-scale
%cd /content/ua-rag-scale
!pip install -q uv && uv sync --frozen -q

from google.colab import files


def download_results():
    !cd $OUT && rm -f /content/results.zip && zip -qr /content/results.zip results
    files.download("/content/results.zip")

In [ ]:
!uv run raglab data --out $OUT

In [ ]:
!uv run raglab embed --out $OUT --device cuda
download_results()

In [ ]:
!uv run raglab finetune --out $OUT --device cuda
download_results()

In [ ]:
!uv run raglab grid --out $OUT
download_results()

In [ ]:
!uv run raglab rerank --out $OUT --device cuda
download_results()

In [ ]:
!uv run raglab grid --out $OUT --latency
!uv run raglab rerank --out $OUT --device cuda --latency
download_results()